In [ ]:
using Pkg
Pkg.activate("..")
using StaticArrays
using Eliashberg
using CairoMakie

println("Eliashberg loaded successfully!")

### 1. Model Setup
Specify the physical parameters and instantiate the `TightBinding{2}` model and `ChargeDensityWave` auxiliary field using the new framework.

In [ ]:
t = 1.0
tp = 0.0   # nearest neighbor hopping
μ = 0.0    # chemical potential
V_Q = 2.0  # bare interaction strength

# Instantiate the model and the CDW field
model = TightBinding{2}(t, tp, μ)
Q = SVector{2, Float64}(pi, pi)
cdw = ChargeDensityWave(Q)

In [ ]:
# Set up the 2D Brillouin Zone Grid and create the EffectiveAction
Nk = 100
points = [SVector{2, Float64}(x, y) for x in range(-pi, pi, length=Nk) for y in range(-pi, pi, length=Nk)]
weights = fill((2*pi/Nk)^2, length(points))

kgrid = KGrid{2}(points, weights)
action = EffectiveAction(model, cdw, kgrid, V_Q)

### 2. Computing the Effective Action
Calculate the non-linear Tr[ln] and the quadratic RPA action vs order parameter $\phi$.

In [ ]:
phi_values = range(0.0, 1.5, length=30)
T_val = 0.05

F_exact_vals = [evaluate(action, phi, ExactTrLn(); T=T_val) for phi in phi_values]
F_RPA_vals   = [evaluate(action, phi, RPA(); T=T_val) for phi in phi_values]

# Solve for the optimal ground state configuration dynamically using Optim
phi_gs_exact = solve_ground_state(action, ExactTrLn(); phi_guess=0.5)
phi_gs_rpa = solve_ground_state(action, RPA(); phi_guess=0.5)

println("Exact Ground state order parameter ϕ = ", phi_gs_exact)
println("RPA Ground state order parameter ϕ = ", phi_gs_rpa)

### 3. Visualization
Plot the action comparison and the 2D band structure utilizing the zero-boilerplate dispersion visualizer.

In [ ]:
# Plot the Free Energy Action (Exact vs RPA)
fig_action = Figure()
ax = Axis(fig_action[1, 1], xlabel = "ϕ", ylabel = "Free Energy F", title="CDW Action: Exact vs RPA")
lines!(ax, phi_values, F_exact_vals, label="Exact Tr[ln]", linewidth=2)
lines!(ax, phi_values, F_RPA_vals, label="RPA", linestyle=:dash, linewidth=2)
hlines!(ax, [0.0], color=:black)
axislegend(ax)
fig_action

In [ ]:
# One-line zero-boilerplate visualization of the native TightBinding 2D model dispersion
fig_bands = visualize_dispersion(model)
fig_bands